In [1]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
from transformers import Wav2Vec2Model
from typing import Dict, List

import librosa
import torchaudio
import torchaudio.transforms as T

import contextlib

from tqdm.auto import tqdm
import pandas as pd
import numpy as np
import ast
from typing import Dict, List

import warnings
import ast

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

In [2]:
import os

os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

In [3]:
# # ? this is for local training

MODEL_PATH = "../../models/best_checkpoint_v1.pth"
WORKING_MODEL_PATH = "../../models/checkpoint.pth"
WORKING_BEST_MODEL_PATH = "../../models/best_checkpoint.pth"

DATASET_PATH = "../../datasets/Quran_ds/Quran_ds/audio/audio/"
DATASET_PATH_1 = "../../datasets/Quran_ds/Quran_ds/audio/audio/"
TRAIN_DS_PATH = "../../datasets/Quran_ds/quran_train_ds_100H_phonemes.csv"
TEST_DS_PATH = "../../datasets/Quran_ds/quran_test_ds_10H_phonemes.csv"

# ? this is for Kaggle training

# MODEL_PATH = "/kaggle/input/datasets/ahmadbannan/my-new-dataset-v1/best_checkpoint.pth"
# WORKING_MODEL_PATH = "/kaggle/working/checkpoint.pth"
# WORKING_BEST_MODEL_PATH = "/kaggle/working/best_checkpoint.pth"


# DATASET_PATH = "/kaggle/input/datasets/omartariq612/quran-reciters/audio/audio/"
# DATASET_PATH_1 = "/kaggle/input/datasets/muhammadbannan/quran-csv-dataset/quran_train_ds_v1.csv"
# TRAIN_DS_PATH = "/kaggle/input/datasets/muhammadbannan/quran-csv-dataset/quran_train_ds_v1.csv"
# TEST_DS_PATH = "/kaggle/input/datasets/muhammadbannan/quran-csv-dataset/quran_test_ds_v1.csv"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

batch_size = 1
sr = 16000
num_epochs = 7

device = torch.device("cpu")


device

device(type='cpu')

In [4]:
BLANK_TOKEN = "<blank>"
SILENT_TOKEN = "<sil>"

IKFAA_LETTERS = [
    "sˤ",
    "ð",
    "θ",
    "k",
    "j",
    "ʃ",
    "q",
    "s",
    "d",
    "tˤ",
    "z",
    "f",
    "t",
    "dˤ",
    "ðˤ",
]

QALQALAA_LETTERS = ["q", "tˤ", "b", "j", "d"]

SHORT_VOWELS: List[str] = ["a", "i", "u"]
LONG_VOWELS: List[str] = ["aa", "ii", "uu"]
TANWEEN: List[str] = ["an", "in", "un"]

SPECIAL_PHONEMES_FOR_TAJWEED = ["n" + l for l in IKFAA_LETTERS]
SPECIAL_PHONEMES_FOR_TAJWEED += ["an" + l for l in IKFAA_LETTERS]
SPECIAL_PHONEMES_FOR_TAJWEED += ["in" + l for l in IKFAA_LETTERS]
SPECIAL_PHONEMES_FOR_TAJWEED += ["un" + l for l in IKFAA_LETTERS]
SPECIAL_PHONEMES_FOR_TAJWEED += [l + "K" for l in QALQALAA_LETTERS]

BASE_CONSONANTS: List[str] = [
    "ʔ",
    "b",
    "t",
    "θ",
    "j",
    "ħ",
    "x",
    "d",
    "ð",
    "r",
    "z",
    "s",
    "ʃ",
    "sˤ",
    "dˤ",
    "tˤ",
    "ðˤ",
    "ʕ",
    "ɣ",
    "f",
    "q",
    "k",
    "l",
    "m",
    "n",
    "h",
    "w",
    "y",
    "T",
]

SPECIAL_PHONEMES_FOR_TAJWEED += ["nn", "mm", "yy", "ww", "rM"]

PHONEMES: List[str] = (
    [
        SILENT_TOKEN,
    ]
    + SPECIAL_PHONEMES_FOR_TAJWEED
    + BASE_CONSONANTS
    + SHORT_VOWELS
    + LONG_VOWELS
    + TANWEEN
)

PHONEMES_CTC: List[str] = [BLANK_TOKEN] + PHONEMES

phoneme_to_id: Dict[str, int] = {p: i for i, p in enumerate(PHONEMES_CTC)}
blank_id: int = phoneme_to_id[BLANK_TOKEN]

### Total duration (TRAIN): 106.5 hours

### Total duration  (TEST): 09.09 hours

In [5]:
@contextlib.contextmanager
def suppress_c_stderr():
    """Redirect C-level stderr to /dev/null — catches libmpg123 warnings."""
    with open(os.devnull, "w") as devnull:
        old_fd = os.dup(2)
        os.dup2(devnull.fileno(), 2)
        try:
            yield
        finally:
            os.dup2(old_fd, 2)
            os.close(old_fd)


def add_noise(signal, noise_level=0.003):
    return signal + noise_level * np.random.randn(len(signal))


def random_gain(signal):
    return signal * np.random.uniform(0.8, 1.2)


def load_waveform(audio_path, sr=16000, training=True):

    # audio_path = audio_path.replace('.wav', '.mp3')

    signal = None

    # ── Step 1: try torchaudio — fast, no temp files ──────────────────────
    try:

        waveform, orig_sr = torchaudio.load(audio_path)

        # Stereo → mono
        if waveform.shape[0] > 1:
            waveform = waveform.mean(dim=0, keepdim=True)

        # Resample if needed
        if orig_sr != sr:
            waveform = T.Resample(orig_freq=orig_sr, new_freq=sr)(waveform)

        signal = waveform.squeeze(0).numpy()

    # ── Step 2: fallback to librosa for corrupt MP3 frames ────────────────
    except Exception:
        try:
            with suppress_c_stderr():
                signal, _ = librosa.load(audio_path, sr=sr, mono=True)
        except Exception as e:
            raise RuntimeError(f"Both loaders failed for: {audio_path} — {e}")

    # ── Step 3: guard against silent/empty output ─────────────────────────
    if signal is None or len(signal) == 0:
        raise RuntimeError(f"Audio is empty: {audio_path}")

    if np.max(np.abs(signal)) < 1e-6:
        raise RuntimeError(f"Audio appears silent or corrupt: {audio_path}")

    # ── Step 4: augmentation (training only) ──────────────────────────────
    if training and np.random.rand() < 0.5:
        signal = add_noise(signal)

    if training and np.random.rand() < 0.5:
        signal = random_gain(signal)

    # ── Step 5: normalize to [-1, 1] ──────────────────────────────────────
    max_val = np.max(np.abs(signal)) + 1e-8
    signal = signal / max_val

    return torch.tensor(signal, dtype=torch.float32)

In [6]:
class DynamicBatchSampler(torch.utils.data.Sampler):
    """
    Groups samples by audio length so each batch contains
    similarly-lengthed ayahs — minimizes padding waste and OOM risk.
    """

    def __init__(self, dataset, max_samples_per_batch, shuffle=True):
        self.dataset = dataset
        self.max_samples_per_batch = max_samples_per_batch
        self.shuffle = shuffle

        # Pre-read waveform lengths from the dataframe
        # (avoids loading audio just to know the length)
        print("Building length index...")
        self.lengths = []
        for idx in range(len(dataset)):
            row = dataset.df.iloc[idx]

            audio_path = ""
            if row["ds_index"] == 1:
                audio_path = os.path.join(dataset.dataset_path, row["path_of_audio"])
            else:
                audio_path = os.path.join(dataset.dataset_path_1, row["path_of_audio"])

            try:
                info = torchaudio.info(audio_path)
                # Resample length if needed
                length = int(info.num_frames * sr / info.sample_rate)
            except Exception:
                length = sr * 10  # fallback: assume 10 seconds
            self.lengths.append(length)
        print(f"Length index built for {len(self.lengths)} samples")

    def __iter__(self):
        batches = self._build_batches()
        for batch in batches:
            yield batch

    def _build_batches(self):
        import random

        indices = list(range(len(self.lengths)))
        if self.shuffle:
            random.shuffle(indices)
        indices.sort(key=lambda i: self.lengths[i])

        batches = []
        current_batch = []
        current_max_len = 0

        for idx in indices:
            length = self.lengths[idx]
            new_max = max(current_max_len, length)
            if (
                current_batch
                and (len(current_batch) + 1) * new_max > self.max_samples_per_batch
            ):
                batches.append(current_batch)
                current_batch = [idx]
                current_max_len = length
            else:
                current_batch.append(idx)
                current_max_len = new_max

        if current_batch:
            batches.append(current_batch)

        if self.shuffle:
            random.shuffle(batches)

        return batches

    def __len__(self):
        # Build once with no shuffle to get stable count
        import random

        indices = list(range(len(self.lengths)))
        indices.sort(key=lambda i: self.lengths[i])
        batches = []
        current_batch = []
        current_max_len = 0
        for idx in indices:
            length = self.lengths[idx]
            new_max = max(current_max_len, length)
            if (
                current_batch
                and (len(current_batch) + 1) * new_max > self.max_samples_per_batch
            ):
                batches.append(current_batch)
                current_batch = [idx]
                current_max_len = length
            else:
                current_batch.append(idx)
                current_max_len = new_max
        if current_batch:
            batches.append(current_batch)
        return len(batches)

In [7]:
class TajweedCTCDataset(Dataset):

    def __init__(
        self, dataframe, dataset_path, dataset_path_1, phoneme_to_id, training=True
    ):
        self.df = dataframe
        self.dataset_path = dataset_path
        self.dataset_path_1 = dataset_path_1
        self.phoneme_to_id = phoneme_to_id
        self.training = training

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        audio_path = ""
        if row["ds_index"] == 1:
            audio_path = os.path.join(self.dataset_path, row["path_of_audio"])
        else:
            audio_path = os.path.join(self.dataset_path_1, row["path_of_audio"])

        waveform = load_waveform(
            audio_path,
            training=self.training,
        )

        phoneme_seq = ast.literal_eval(row["phonemes"])
        target_ids = [
            self.phoneme_to_id[p] for p in phoneme_seq if p in self.phoneme_to_id
        ]

        return (
            waveform,
            torch.tensor(target_ids, dtype=torch.long),
            waveform.shape[0],
            len(target_ids),
        )

In [8]:
def ctc_collate(batch):
    waveforms, targets, input_lengths, target_lengths = zip(*batch)
    padded_waveforms = pad_sequence(waveforms, batch_first=True)
    return (
        padded_waveforms,
        torch.cat(targets),
        torch.tensor(input_lengths, dtype=torch.long),
        torch.tensor(target_lengths, dtype=torch.long),
    )

In [9]:
# Load wav2vec2 — frozen feature extractor, runs on GPU
wav2vec2_model = Wav2Vec2Model.from_pretrained(
    "facebook/wav2vec2-large-xlsr-53", ignore_mismatched_sizes=True
)

# Freeze everything first
for param in wav2vec2_model.parameters():
    param.requires_grad = False

# Unfreeze last 4 transformer layers
for param in wav2vec2_model.encoder.layers[-6:].parameters():
    param.requires_grad = True

WAV2VEC2_STRIDE = 320  # 16kHz / 320 = 50 frames/sec


def get_feature_lengths(input_lengths):
    """Convert raw sample counts → wav2vec2 output frame counts."""
    return torch.clamp((input_lengths - 400) // WAV2VEC2_STRIDE + 1, min=1)

Some weights of Wav2Vec2Model were not initialized from the model checkpoint at facebook/wav2vec2-large-xlsr-53 and are newly initialized: ['wav2vec2.encoder.pos_conv_embed.conv.parametrizations.weight.original0', 'wav2vec2.encoder.pos_conv_embed.conv.parametrizations.weight.original1']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [10]:
class SpecAugment(nn.Module):
    """
    Conservative SpecAugment tuned for Tajweed:
    - Small time masks to preserve madd duration information
    - Small freq masks to preserve emphatic/nasal phoneme signatures
    - Multiple masks instead of one large one
    """

    def __init__(
        self,
        time_mask_max=8,  # max 80ms erased — safe for short vowels (~150ms)
        freq_mask_max=10,  # max 10/128 channels — preserves spectral shape
        num_time_masks=2,  # two small time masks instead of one big one
        num_freq_masks=2,  # two small freq masks
    ):
        super().__init__()
        self.time_mask_max = time_mask_max
        self.freq_mask_max = freq_mask_max
        self.num_time_masks = num_time_masks
        self.num_freq_masks = num_freq_masks

    def forward(self, x):
        # x: (B, T, C)
        if not self.training:
            return x

        B, T, C = x.shape
        x = x.clone()

        # Multiple small time masks
        for _ in range(self.num_time_masks):
            t = np.random.randint(1, self.time_mask_max + 1)
            t0 = np.random.randint(0, max(1, T - t))
            x[:, t0 : t0 + t, :] = 0

        # Multiple small frequency masks
        for _ in range(self.num_freq_masks):
            f = np.random.randint(1, self.freq_mask_max + 1)
            f0 = np.random.randint(0, max(1, C - f))
            x[:, :, f0 : f0 + f] = 0

        return x

In [11]:
# ── Conformer building blocks ──────────────────────────────────────────────


class FeedForwardModule(nn.Module):
    """The two feed-forward half-steps that sandwich each Conformer block."""

    def __init__(self, dim, expansion=4, dropout=0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.LayerNorm(dim),
            nn.Linear(dim, dim * expansion),
            nn.SiLU(),  # Swish activation
            nn.Dropout(dropout),
            nn.Linear(dim * expansion, dim),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return x + 0.5 * self.net(x)  # 0.5 scaling from original paper


class ConvolutionModule(nn.Module):
    """Local feature extraction — captures phoneme-level patterns."""

    def __init__(self, dim, kernel_size=31, dropout=0.1):
        super().__init__()
        # kernel_size must be odd for same-length output
        assert kernel_size % 2 == 1
        self.net = nn.Sequential(
            nn.LayerNorm(dim),
            # Pointwise expand
            nn.Linear(dim, dim * 2),
            nn.GLU(dim=-1),  # gates half the channels → back to dim
            # Depthwise conv — operates per-channel, captures local timing
            nn.Conv1d(dim, dim, kernel_size, padding=kernel_size // 2, groups=dim),
            nn.BatchNorm1d(dim),
            nn.SiLU(),
            # Pointwise project back
            nn.Linear(dim, dim),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        # x: (B, T, dim)
        residual = x
        x = self.net[0](x)  # LayerNorm
        x = self.net[1](x)  # Linear expand
        x = self.net[2](x)  # GLU
        x = x.transpose(1, 2)  # → (B, dim, T) for Conv1d
        x = self.net[3](x)  # Depthwise Conv
        x = self.net[4](x)  # BatchNorm
        x = self.net[5](x)  # SiLU
        x = x.transpose(1, 2)  # → (B, T, dim)
        x = self.net[6](x)  # Linear project
        x = self.net[7](x)  # Dropout
        return residual + x


class ConformerBlock(nn.Module):
    """
    Full Conformer block:
    FF → Self-Attention → Convolution → FF → LayerNorm
    """

    def __init__(self, dim, num_heads=4, kernel_size=31, ff_expansion=4, dropout=0.1):
        super().__init__()

        self.ff1 = FeedForwardModule(dim, ff_expansion, dropout)
        self.attn_norm = nn.LayerNorm(dim)
        self.attn = nn.MultiheadAttention(
            dim, num_heads, dropout=dropout, batch_first=True
        )
        self.attn_drop = nn.Dropout(dropout)
        self.conv = ConvolutionModule(dim, kernel_size, dropout)
        self.ff2 = FeedForwardModule(dim, ff_expansion, dropout)
        self.norm_out = nn.LayerNorm(dim)

    def forward(self, x, key_padding_mask=None):
        # FF half-step
        x = self.ff1(x)

        # Self-attention
        residual = x
        x_norm = self.attn_norm(x)
        x_attn, _ = self.attn(x_norm, x_norm, x_norm, key_padding_mask=key_padding_mask)
        x = residual + self.attn_drop(x_attn)

        # Convolution
        x = self.conv(x)

        # FF half-step
        x = self.ff2(x)

        return self.norm_out(x)


# ── Full model ─────────────────────────────────────────────────────────────


class Wav2Vec2Conformer_CTC(nn.Module):
    """
    wav2vec2-large-xlsr-53 (frozen) → Linear projection →
    4x Conformer blocks → CTC classifier
    """

    def __init__(
        self,
        num_classes,
        conformer_dim=256,  # internal dim of Conformer
        num_heads=4,
        num_blocks=4,  # 4 blocks is enough — keeps memory manageable
        kernel_size=31,  # 31 frames = ~620ms local context
        dropout=0.1,
    ):
        super().__init__()

        # Project wav2vec2 output (1024) down to conformer_dim
        self.input_projection = nn.Sequential(
            nn.Linear(1024, conformer_dim),
            nn.Dropout(dropout),
        )

        self.conformer_blocks = nn.ModuleList(
            [
                ConformerBlock(
                    dim=conformer_dim,
                    num_heads=num_heads,
                    kernel_size=kernel_size,
                    dropout=dropout,
                )
                for _ in range(num_blocks)
            ]
        )

        self.classifier = nn.Linear(conformer_dim, num_classes)

    def forward(self, features, feat_lengths):
        # features: (B, T', 1024) from wav2vec2
        # feat_lengths: (B,) actual frame counts before padding

        # Project to conformer dim
        x = self.input_projection(features)  # (B, T', conformer_dim)

        # Build padding mask for attention
        # True = position is padding (should be ignored)
        B, T, _ = x.shape
        mask = torch.arange(T, device=x.device).unsqueeze(0) >= feat_lengths.unsqueeze(
            1
        )  # (B, T)

        # Pass through Conformer blocks
        for block in self.conformer_blocks:
            x = block(x, key_padding_mask=mask)

        # CTC classifier
        logits = self.classifier(x)  # (B, T', num_classes)
        logits = logits.permute(1, 0, 2)  # (T', B, num_classes) for CTC

        return logits

In [12]:
class ASRModel(torch.nn.Module):

    def __init__(self, wav2vec2, conformer, spec_augment):
        super().__init__()
        self.wav2vec2 = wav2vec2
        self.conformer = conformer
        self.spec_augment = spec_augment

    def forward(self, waveforms, input_lengths, feat_lengths):
        self.wav2vec2.feature_extractor.eval()

        B, T = waveforms.shape
        attention_mask = torch.zeros(B, T, dtype=torch.long, device=waveforms.device)
        for i, length in enumerate(input_lengths):
            attention_mask[i, :length] = 1

        outputs = self.wav2vec2(waveforms, attention_mask=attention_mask)
        features = outputs.last_hidden_state  # (B, T', 1024)

        # Zero out contaminated padding frames from CNN
        B, T_feat, D = features.shape
        feat_mask = torch.arange(T_feat, device=features.device).unsqueeze(
            0
        ) < feat_lengths.unsqueeze(
            1
        )  # (B, T')
        features = features * feat_mask.unsqueeze(-1).float()  # zero padding frames

        if self.training:
            features = self.spec_augment(features)

        logits = self.conformer(features, feat_lengths)
        return logits

In [13]:
train_df = pd.read_csv(TRAIN_DS_PATH)
val_df = pd.read_csv(TEST_DS_PATH)

In [14]:
train_dataset = TajweedCTCDataset(
    dataframe=train_df,
    training=True,
    dataset_path=DATASET_PATH,
    dataset_path_1=DATASET_PATH_1,
    phoneme_to_id=phoneme_to_id,
)
val_dataset = TajweedCTCDataset(
    dataframe=val_df,
    training=False,
    dataset_path=DATASET_PATH,
    dataset_path_1=DATASET_PATH_1,
    phoneme_to_id=phoneme_to_id,
)


MAX_TOKENS = batch_size * 16000 * 20

train_sampler = DynamicBatchSampler(
    train_dataset,
    max_samples_per_batch=MAX_TOKENS,
    shuffle=True,
)

val_sampler = DynamicBatchSampler(
    val_dataset,
    max_samples_per_batch=MAX_TOKENS,
    shuffle=False,
)

train_loader = DataLoader(
    train_dataset,
    batch_sampler=train_sampler,
    # batch_size=batch_size,
    collate_fn=ctc_collate,
    num_workers=2,
    pin_memory=True,
)

val_loader = DataLoader(
    val_dataset,
    batch_sampler=val_sampler,
    # batch_size=batch_size,
    collate_fn=ctc_collate,
    num_workers=2,
    pin_memory=True,
)

Building length index...
Length index built for 34427 samples
Building length index...
Length index built for 2486 samples


In [15]:
next(enumerate(train_loader))

(0,
 [tensor([[ 0.0012, -0.0060, -0.0040,  ...,  0.0055,  0.0161,  0.0151]]),
  tensor([ 98, 101,  72, 102,  80,  99, 101,  95,  93, 103,  71, 103,  73, 102,
           93, 101,  94,  94, 101,  80, 105,  96, 101,  93, 101,  91, 101,  71,
          106,  94, 102,  99, 103,  75,  73, 102,  74, 106,  93, 101,  72, 101,
           98,  99, 101,  62,  74, 103,  94, 106,  93, 101,  72, 101,  98,  99,
          103,  78,  81, 102,  76, 106,  93, 101,  98, 101,  99, 101,  95,  93,
          103,  71, 106,  96, 101,  98, 101,  99, 101,  95,  93, 103,  71, 103,
           94,  94, 104,  97, 103,  98, 101,  94,  94, 104,  97, 103,  78, 101,
           99,  71, 103,  94,  95, 104,  93, 102,  81, 105,  96]),
  tensor([231550]),
  tensor([109])])

In [16]:
def save_checkpoint(
    model, optimizer, epoch, loss, best_val_loss, epochs_no_improve, scheduler, path
):
    # Unwrap DataParallel before saving
    model_state = (
        model.module.state_dict()
        if isinstance(model, torch.nn.DataParallel)
        else model.state_dict()
    )
    torch.save(
        {
            "epoch": epoch,
            "model_state": model_state,
            "optimizer_state": optimizer.state_dict(),
            "loss": loss,
            "best_val_loss": best_val_loss,
            "epochs_no_improve": epochs_no_improve,
            "scheduler_state": scheduler.state_dict(),
        },
        path,
    )


def load_checkpoint(path):
    checkpoint = torch.load(path)
    return checkpoint

In [17]:
from accelerate import Accelerator
from accelerate.utils import DistributedDataParallelKwargs

ddp_kwargs = DistributedDataParallelKwargs(find_unused_parameters=True)
accelerator = Accelerator(mixed_precision="fp16", kwargs_handlers=[ddp_kwargs])

print(f"Using device: {device}")
print(f"Num processes: {accelerator.num_processes}")

conformer_model = Wav2Vec2Conformer_CTC(
    num_classes=len(phoneme_to_id),
    conformer_dim=256,
    num_heads=4,
    num_blocks=4,
    kernel_size=31,
    dropout=0.15,
)

model = ASRModel(
    wav2vec2_model,
    conformer_model,
    spec_augment=SpecAugment(
        time_mask_max=6,
        freq_mask_max=6,
        num_time_masks=2,
        num_freq_masks=2,
    ),
)

ctc_loss = nn.CTCLoss(blank=blank_id, zero_infinity=True, reduction="mean")


# Only optimize the Conformer — wav2vec2 has no gradients
optimizer = torch.optim.AdamW(
    [
        {"params": model.wav2vec2.encoder.parameters(), "lr": 1e-5},
        {"params": model.conformer.parameters(), "lr": 1e-5},
    ],
    weight_decay=0.01,
)


scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    factor=0.5,
    patience=3,
    min_lr=1e-6,
)


# # Let accelerate handle everything
# model, optimizer, train_loader, val_loader = accelerator.prepare(
#     model, optimizer, train_loader, val_loader
# )


best_val_loss = float("inf")
epochs_no_improve = 0

if os.path.exists(MODEL_PATH):
    print("Loading checkpoint...")

    ckpt = torch.load(MODEL_PATH, map_location=device)

    model.load_state_dict(ckpt["model_state"])
    optimizer.load_state_dict(ckpt["optimizer_state"])
    scheduler.load_state_dict(ckpt["scheduler_state"])

    best_val_loss = ckpt["best_val_loss"]
    epochs_no_improve = ckpt["epochs_no_improve"]

    start_epoch = ckpt["epoch"] + 1
    print("done loading")
else:
    print("Starting fresh training")
    best_val_loss= float('inf')
    epochs_no_improve=0
    start_epoch = 0

Using device: cpu
Num processes: 1
Loading checkpoint...
done loading


In [18]:
model

ASRModel(
  (wav2vec2): Wav2Vec2Model(
    (feature_extractor): Wav2Vec2FeatureEncoder(
      (conv_layers): ModuleList(
        (0): Wav2Vec2LayerNormConvLayer(
          (conv): Conv1d(1, 512, kernel_size=(10,), stride=(5,))
          (layer_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
          (activation): GELUActivation()
        )
        (1-4): 4 x Wav2Vec2LayerNormConvLayer(
          (conv): Conv1d(512, 512, kernel_size=(3,), stride=(2,))
          (layer_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
          (activation): GELUActivation()
        )
        (5-6): 2 x Wav2Vec2LayerNormConvLayer(
          (conv): Conv1d(512, 512, kernel_size=(2,), stride=(2,))
          (layer_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
          (activation): GELUActivation()
        )
      )
    )
    (feature_projection): Wav2Vec2FeatureProjection(
      (layer_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
      (projection):

In [19]:
def ctc_decode(frame_preds, blank_id):

    decoded = []
    prev = None

    for p in frame_preds:

        p = int(p)

        if p != blank_id and p != prev:
            decoded.append(p)

        prev = p

    return decoded

In [20]:
def compute_per(logits, targets, feat_lengths, target_lengths, blank_id):
    """Phoneme Error Rate — lower is better."""
    pred = torch.argmax(logits, dim=2).permute(1, 0).cpu()  # (B, T)
    feat_lengths = feat_lengths.cpu()
    target_lengths = target_lengths.cpu()
    targets = targets.cpu()

    total_errors = 0
    total_phonemes = 0
    offset = 0

    for i in range(pred.shape[0]):
        decoded = ctc_decode(pred[i, : feat_lengths[i]].tolist(), blank_id)
        length = target_lengths[i].item()
        ref = targets[offset : offset + length].tolist()
        offset += length

        # Simple edit distance
        n, m = len(ref), len(decoded)
        dp = list(range(m + 1))
        for r in ref:
            new_dp = [dp[0] + 1]
            for j, h in enumerate(decoded):
                new_dp.append(
                    min(dp[j + 1] + 1, new_dp[-1] + 1, dp[j] + (0 if r == h else 1))
                )
            dp = new_dp
        total_errors += dp[m]
        total_phonemes += n

    return total_errors / max(total_phonemes, 1)

In [21]:
def train_model(
    model,
    train_loader,
    val_loader,
    ctc_loss,
    optimizer,
    scheduler,
    accelerator,
    epochs=30,
    patience=4,
    best_val_loss=float("inf"),
    epochs_no_improve=0,
):
    for epoch in range(epochs):
        model.train()
        train_loss = 0

        for waveforms, targets, input_lengths, target_lengths in tqdm(train_loader):
            feat_lengths = get_feature_lengths(input_lengths)
            optimizer.zero_grad(set_to_none=True)

            logits = model(waveforms, input_lengths, feat_lengths)
            log_probs = torch.log_softmax(logits, dim=2)
            loss = ctc_loss(log_probs.float(), targets, feat_lengths, target_lengths)

            accelerator.backward(loss)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            optimizer.step()

            train_loss += loss.item()

        avg_train_loss = train_loss / len(train_loader)

        # ── Validate ───────────────────────────────────────────────────
        model.eval()
        val_loss = 0
        per = 0

        with torch.no_grad():
            for waveforms, targets, input_lengths, target_lengths in tqdm(val_loader):
                feat_lengths = get_feature_lengths(input_lengths)

                logits = model(waveforms, input_lengths, feat_lengths)
                log_probs = torch.log_softmax(logits, dim=2)
                loss = ctc_loss(
                    log_probs.float(), targets, feat_lengths, target_lengths
                )

                val_loss += loss.item()
                per += compute_per(
                    logits, targets, feat_lengths, target_lengths, blank_id
                )

        avg_val_loss = val_loss / len(val_loader)
        per = per / len(val_loader)
        scheduler.step(avg_val_loss)

        unwrapped = accelerator.unwrap_model(model)

        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            epochs_no_improve = 0
            save_checkpoint(
                unwrapped,
                optimizer,
                epoch,
                avg_val_loss,
                best_val_loss,
                epochs_no_improve,
                scheduler,
                WORKING_BEST_MODEL_PATH,
            )
            print("✅ Best model saved")
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= patience:
                print(f"⏹ Early stopping at epoch {epoch+1}")
                break

        save_checkpoint(
            unwrapped,
            optimizer,
            epoch,
            avg_val_loss,
            best_val_loss,
            epochs_no_improve,
            scheduler,
            WORKING_MODEL_PATH,
        )

        print(f"\nEpoch {epoch+1}/{epochs}")
        print(f"  Train Loss : {avg_train_loss:.4f}")
        print(f"  Val Loss   : {avg_val_loss:.4f}")
        print(f"  Val PER    : {per:.4f}")
        print("-" * 40)

In [22]:
# train_model(
#     model=model,
#     train_loader=train_loader,
#     val_loader=val_loader,
#     ctc_loss=ctc_loss,
#     optimizer=optimizer,
#     scheduler=scheduler,
#     accelerator=accelerator,
#     epochs=num_epochs,
#     patience=6,
#     best_val_loss=best_val_loss,
#     epochs_no_improve=epochs_no_improve,
# )

### eval model

In [23]:
# ckpt = torch.load(MODEL_PATH, map_location=device)

# model.load_state_dict(ckpt["model_state"])

id_to_phoneme = {v: k for k, v in phoneme_to_id.items()}

In [24]:
def custom_ctc_decode(frame_preds, blank_id):
    decoded = []
    prev = None
    count = 0

    for p in frame_preds:
        p = int(p)

        if p != blank_id and p != prev:
            if decoded:
                decoded[-1]["count"] = count

            decoded.append({"char": p, "count": 0})
            count = 0

        prev = p
        count += 1

    if decoded:
        decoded[-1]["count"] = count

    return decoded

In [48]:
def predict_phonemes(path:str,device):
    waveforms = load_waveform(path, training=False)
    waveforms = waveforms.unsqueeze(0).to(device)  # (1, T)
    model.eval()
    with torch.no_grad():
        input_lengths = torch.tensor([waveforms.shape[1]], dtype=torch.long,device=device)
        feat_lengths = get_feature_lengths(input_lengths)
        logits = model(waveforms, input_lengths, feat_lengths)
        preds = torch.argmax(logits, dim=2).permute(1, 0).cpu()  # (B, T)
        decoded_preds = custom_ctc_decode(preds[0], blank_id)
        pred_phonemes = [id_to_phoneme[int(p['char'])] for p in decoded_preds]
    return pred_phonemes

In [25]:
# waveforms = load_waveform('../1.mp3', training=False)
# waveforms = load_waveform('../2.mp3', training=False)
# waveforms = load_waveform('../3.mp3', training=False)


waveforms = load_waveform("../../datasets/ayah_dataset/ayah_dataset/kaggle/working/Sulalatun_Nimah_mp3/Sulalatun_Ni'mah_mp3/018-001.mp3", training=False)
# waveforms = load_waveform("../../datasets/ayah_dataset/kaggle/working/Sulalatun_Nimah_mp3/Sulalatun_Ni'mah_mp3/018-002.mp3", training=False)
# waveforms = load_waveform("../../datasets/ayah_dataset/kaggle/working/Husary/audio/Husary_64kbps/058003.mp3", training=False)
# waveforms = load_waveform("../../datasets/ayah_dataset/kaggle/working/Husary/audio/Husary_64kbps/058003.mp3", training=False)

# waveforms = load_waveform("../002004.mp3", training=False)
# waveforms = load_waveform('../001002.mp3', training=False)


waveforms = waveforms.unsqueeze(0).to(device)  # (1, T)

print("waveforms", waveforms.shape)

model.eval()
with torch.no_grad():
    input_lengths = torch.tensor([waveforms.shape[1]], dtype=torch.long,device=device)
    feat_lengths = get_feature_lengths(input_lengths)

    logits = model(waveforms, input_lengths, feat_lengths)

    print(logits.shape)

    preds = torch.argmax(logits, dim=2).permute(1, 0).cpu()  # (B, T)

    print("preds", preds.shape)

    decoded_preds = custom_ctc_decode(preds[0], blank_id)

    # print("decoded_preds", decoded_preds)

    pred_phonemes = [{'char':id_to_phoneme[int(p['char'])],'count':p['count']} for p in decoded_preds]

    pred_text = [id_to_phoneme[int(p['char'])] for p in decoded_preds]


print("Predicted phonemes:", pred_phonemes)
print("Predicted text:", pred_text)

waveforms torch.Size([1, 402404])
torch.Size([1257, 1, 110])
preds torch.Size([1, 1257])
Predicted phonemes: [{'char': 'ʔ', 'count': 9}, {'char': 'l', 'count': 18}, {'char': 'ħ', 'count': 2}, {'char': 'a', 'count': 10}, {'char': 'm', 'count': 6}, {'char': 'm', 'count': 10}, {'char': 'd', 'count': 2}, {'char': 'u', 'count': 11}, {'char': 'l', 'count': 3}, {'char': 'i', 'count': 7}, {'char': 'l', 'count': 15}, {'char': 'l', 'count': 17}, {'char': 'aa', 'count': 11}, {'char': 'l', 'count': 3}, {'char': 'i', 'count': 9}, {'char': 'l', 'count': 12}, {'char': 'l', 'count': 2}, {'char': 'a', 'count': 13}, {'char': 'ð', 'count': 68}, {'char': 'ii', 'count': 10}, {'char': 'ʔ', 'count': 2}, {'char': 'a', 'count': 74}, {'char': 'nz', 'count': 10}, {'char': 'z', 'count': 2}, {'char': 'a', 'count': 12}, {'char': 'l', 'count': 2}, {'char': 'a', 'count': 10}, {'char': 'ʕ', 'count': 2}, {'char': 'a', 'count': 11}, {'char': 'l', 'count': 16}, {'char': 'aa', 'count': 11}, {'char': 'ʕ', 'count': 2}, {'ch

In [39]:
from __future__ import annotations

import difflib
from dataclasses import dataclass
from typing import List, Optional, Tuple

_NORM = {
    "aa": "a",
    "ii": "i",
    "uu": "u",
    "mm": "m",
    "nn": "n",
    "bb": "b",
    "nz": "n",
    "bK": "b",
    "jK": "j",
    "dK": "d",
}

def _norm(p: str) -> str:
    return _NORM.get(p, p)

def _similar(a: str, b: str) -> bool:
    return a == b or _norm(a) == _norm(b)

def _align(
    generated: List[str],
    label: List[str],
    match: float = 1.0,
    mismatch: float = -0.4,
    gap: float = -0.3,
) -> Tuple[List[str], List[str], List[str]]:
    G, L = len(generated), len(label)
    dp = [[0.0] * (L + 1) for _ in range(G + 1)]
    for i in range(1, G + 1):
        dp[i][0] = dp[i - 1][0] + gap
    for j in range(1, L + 1):
        dp[0][j] = dp[0][j - 1] + gap

    for i in range(1, G + 1):
        for j in range(1, L + 1):
            sc = match if _similar(generated[i - 1], label[j - 1]) else mismatch
            dp[i][j] = max(
                dp[i - 1][j - 1] + sc,
                dp[i - 1][j] + gap,
                dp[i][j - 1] + gap,
            )

    gen_a, lbl_a, ops = [], [], []
    i, j = G, L
    while i > 0 or j > 0:
        if i > 0 and j > 0:
            sc = match if _similar(generated[i - 1], label[j - 1]) else mismatch
            if dp[i][j] == dp[i - 1][j - 1] + sc:
                gen_a.append(generated[i - 1])
                lbl_a.append(label[j - 1])
                ops.append("M" if _similar(generated[i - 1], label[j - 1]) else "X")
                i -= 1
                j -= 1
                continue
        if i > 0 and (j == 0 or dp[i][j] == dp[i - 1][j] + gap):
            gen_a.append(generated[i - 1])
            lbl_a.append("-")
            ops.append("I")
            i -= 1
        else:
            gen_a.append("-")
            lbl_a.append(label[j - 1])
            ops.append("D")
            j -= 1

    for lst in (gen_a, lbl_a, ops):
        lst.reverse()
    return gen_a, lbl_a, ops

def _collect_runs(gen_a: List[str], ops: List[str]) -> List[dict]:
    runs = []
    label_consumed = -1
    i = 0
    while i < len(ops):
        if ops[i] in ("M", "X", "D"):
            label_consumed += 1
            i += 1
        elif ops[i] == "I":
            run_phonemes = []
            while i < len(ops) and ops[i] == "I":
                run_phonemes.append(gen_a[i])
                i += 1
            runs.append({"phonemes": run_phonemes, "anchor": label_consumed})
        else:
            i += 1
    return runs

def _merge_runs(runs: List[dict], gap_tolerance: int = 4) -> List[dict]:
    if not runs:
        return []
    merged = [dict(phonemes=list(runs[0]["phonemes"]), anchor=runs[0]["anchor"])]
    for r in runs[1:]:
        last = merged[-1]
        if r["anchor"] - last["anchor"] <= gap_tolerance:
            last["phonemes"] = last["phonemes"] + r["phonemes"]
        else:
            merged.append(dict(phonemes=list(r["phonemes"]), anchor=r["anchor"]))
    return merged


def _similarity(a: List[str], b: List[str]) -> float:
    return difflib.SequenceMatcher(None, a, b).ratio()


def _find_label_slice(
    candidate_gen: List[str],  # noisy phonemes from generated (used only for matching)
    label: List[str],
    anchor: int,
    look_back: int = 100,
) -> Tuple[float, int, int]:
    """
    Find the label sub-sequence (label[s:e]) that best matches `candidate_gen`.
    Only searches in label[max(0, anchor-look_back) : anchor+1].
    Returns (similarity, s, e).
    """
    clen = len(candidate_gen)
    if clen == 0:
        return 0.0, 0, 0

    region_s = max(0, anchor - look_back + 1)
    region_e = anchor + 1

    best_sim, best_s, best_e = 0.0, 0, 0
    lo = max(2, int(clen * 0.70))
    hi = min(len(label), int(clen * 1.30) + 1)

    for wlen in range(lo, hi + 1):
        for s in range(region_s, region_e - wlen + 2):
            e = s + wlen
            if e > len(label):
                break
            sim = _similarity(candidate_gen, label[s:e])
            if sim > best_sim:
                best_sim, best_s, best_e = sim, s, e

    return best_sim, best_s, best_e

@dataclass
class RepetitionInfo:
    anchor: int  # inserted after label[anchor]
    label_slice: List[str]  # exact label phonemes that are repeated (clean)
    gen_phonemes: List[str]  # what the model generated (noisy reference)
    similarity: float
    label_start: int  # label[label_start:label_end] = label_slice
    label_end: int


@dataclass
class FixResult:
    corrected_label: List[str]
    repetitions: List[RepetitionInfo]
    report: str


def fix_label(
    generated: List[str],
    label: List[str],
    similarity_threshold: float = 0.60,
    min_run_length: int = 5,
    look_back: int = 100,
    merge_gap: int = 4,
) -> FixResult:
    """
    Detect repetitions in `generated` and insert the corresponding clean
    LABEL slices into `label` at the correct positions.

    Parameters
    ----------
    generated            : phoneme list from demo ASR model (may contain errors)
    label                : original ground-truth phoneme list
    similarity_threshold : min SequenceMatcher ratio to accept a repetition
    min_run_length       : discard insert runs shorter than this
    look_back            : search window size (label phonemes before anchor)
    merge_gap            : merge adjacent insert runs within this many positions
    """
    gen_a, lbl_a, ops = _align(generated, label)

    raw_runs = _collect_runs(gen_a, ops)
    filtered = [r for r in raw_runs if len(r["phonemes"]) >= min_run_length]
    merged = _merge_runs(filtered, gap_tolerance=merge_gap)

    accepted: List[RepetitionInfo] = []
    for run in merged:
        anchor = run["anchor"]
        gen_ph = run["phonemes"]
        sim, s, e = _find_label_slice(gen_ph, label, anchor, look_back)
        if sim >= similarity_threshold:
            accepted.append(
                RepetitionInfo(
                    anchor=anchor,
                    label_slice=label[s:e],  # ← clean label phonemes, not generated
                    gen_phonemes=gen_ph,
                    similarity=sim,
                    label_start=s,
                    label_end=e,
                )
            )

    accepted.sort(key=lambda r: r.anchor)

    # Build corrected label
    corrected: List[str] = []
    insertion_map = {r.anchor: r for r in accepted}

    if -1 in insertion_map:
        corrected.append("<sil>")
        corrected.extend(insertion_map[-1].label_slice)

    for i, ph in enumerate(label):
        corrected.append(ph)
        if i in insertion_map:
            corrected.append("<sil>")  # silence before the repetition
            corrected.extend(insertion_map[i].label_slice)

    # Report
    lines = [
        "=" * 60,
        "PHONEME LABEL FIX REPORT",
        "=" * 60,
        f"Original label length : {len(label)}",
        f"Generated seq length  : {len(generated)}",
        f"Raw insert runs       : {len(raw_runs)}",
        f"After length filter   : {len(filtered)}",
        f"After merging         : {len(merged)}",
        f"Accepted (sim ≥ {similarity_threshold:.0%}) : {len(accepted)}",
        "",
    ]
    if accepted:
        lines.append("Accepted repetitions (from LABEL, not generated):")
        for k, r in enumerate(accepted, 1):
            preview = r.label_slice[:12]
            ellipsis = "…" if len(r.label_slice) > 12 else ""
            lines.append(
                f"  [{k}] after label[{r.anchor:3d}] | "
                f"sim={r.similarity:.2f} | "
                f"label[{r.label_start}:{r.label_end}] (len={r.label_end-r.label_start}) | "
                f"{preview}{ellipsis}"
            )
    lines += [
        "",
        f"Corrected label length: {len(corrected)}",
        "=" * 60,
    ]
    return FixResult(
        corrected_label=corrected, repetitions=accepted, report="\n".join(lines)
    )

In [ ]:
# train_df = pd.read_csv("../../datasets/Quran_ds/quran_train_ds_phonemes.csv")

# for row in tqdm(train_df.itertuples(), total=len(train_df)):
#     audio_path = os.path.join("../../datasets/Quran_ds/Quran_ds/audio/audio", row.path_of_audio)
#     generated_phonemes = predict_phonemes(audio_path, device)
#     label_phonemes = ast.literal_eval(row.phonemes)
#     fix_result = fix_label(
#         generated=generated_phonemes,
#         label=label_phonemes,
#         similarity_threshold=0.60,
#         min_run_length=5,
#         merge_gap=4,
#     )

#     if(len(label_phonemes) != len(fix_result.corrected_label)):
#         train_df.at[row.Index, "phonemes"] = str(fix_result.corrected_label)

# pd.to_csv(train_df, "kaggle/working/quran_train_ds_phonemes", index=False)

In [26]:
# TEST_DATASET_PATH='../../datasets/QDAT_Quran_DS/FINAL_SOUND'
# files_names=[]
# for path,folders , files in os.walk(TEST_DATASET_PATH):
#     files_names.extend(files)

# targets=['qaa', 'luu', 'laa', 'ʕi', 'l', 'ma', 'la', 'naa', 'ʔi', 'n', 'na', 'ka', 'ʔa', 'n', 'ta', 'ʕa', 'l', 'laa', 'mu', 'l', 'ɣu', 'yuu', 'bi']

# per = 0

# for item in tqdm(files_names):

#     file_path = os.path.join(TEST_DATASET_PATH, item).split('/')

#     file_path[-1] = file_name

#     file_path =  '/'.join(file_path)

#     waveforms= load_waveform(file_path, training=False)
#     waveforms = waveforms.unsqueeze(0).to(device)  # (1, T)

#     model.eval()
#     with torch.no_grad():
#         input_lengths = torch.tensor([waveforms.shape[1]], dtype=torch.long)
#         feat_lengths = get_feature_lengths(input_lengths)
#         logits = model(waveforms,input_lengths, feat_lengths)

#         preds = torch.argmax(logits, dim=2).permute(1, 0).cpu()  # (B, T)

#         decoded_preds = ctc_decode(preds[0],blank_id)

#         pred_phonemes = [id_to_phoneme[int(p)] for p in decoded_preds]

#         target_ids = torch.tensor([phoneme_to_id[p] for p in targets if p in phoneme_to_id], dtype=torch.long)
#         per += compute_per(logits, target_ids, feat_lengths, torch.tensor([len(target_ids)]), blank_id)

# print(f"Average Phoneme Error Rate across test set: {per/len(files_names):.4f}")

### Average Phoneme Error Rate across test set (QDAT) : 0.1795

In [27]:
# TEST_DATASET_PATH='../../datasets/ayah_dataset/ayat_with_phonemes.csv'


# test_df= pd.read_csv(TEST_DATASET_PATH)


# test_df= test_df.where(test_df['reciter_name']=="Sulalatun_Nimah")

# test_df= test_df.dropna()

# per = 0

# for item in tqdm(test_df.itertuples()):

#     file_path=item.path_of_audio

#     file_path = '../../datasets/ayah_dataset/ayah_dataset/'+file_path

#     waveforms= load_waveform(file_path, training=False)
#     waveforms = waveforms.unsqueeze(0).to(device)  # (1, T)

#     model.eval()
#     with torch.no_grad():
#         input_lengths = torch.tensor([waveforms.shape[1]], dtype=torch.long)
#         feat_lengths = get_feature_lengths(input_lengths)
#         logits = model(waveforms,input_lengths, feat_lengths)

#         preds = torch.argmax(logits, dim=2).permute(1, 0).cpu()  # (B, T)

#         decoded_preds = ctc_decode(preds[0],blank_id)

#         phonemes, _ = ast.literal_eval(item.phonemes)

#         target_ids = torch.tensor([phoneme_to_id[p] for p in phonemes if p in phoneme_to_id], dtype=torch.long)

#         per += compute_per(logits, target_ids, feat_lengths, torch.tensor([len(target_ids)]), blank_id)

# print(f"Average Phoneme Error Rate across test set: {per/len(test_df):.4f}")